# OnKith Model V2 - DeBERTa-v3-xsmall

One linear, restartable run: build the data, train `microsoft/deberta-v3-xsmall`
on the canonical 63-label ontology, evaluate on six separate sets, export ONNX
and INT8, and produce the deployment package.

**Ontology.** 31 entity types, 63 BIO labels. Relative to the TinyBERT 67-label
head: `ALLERGIES`, `USERNAME` and `WEIGHT` are removed, `HEIGHT` stays out
(it was never in the head), and `PASSWORD` is added. `PASSWORD` is admitted only
for genuine conversational credential disclosure - see
`privacy_data_v2/password_rules.py`.

**Data.** AI4Privacy English is the base. Gretel, Nemotron, Kiji and the Saudi
generator are *targeted gap-fillers*: rows are selected because they strengthen
a weak label, never concatenated wholesale. Once a row is selected every
correctly mapped span in it is kept.

**Evaluation sets, in the order this notebook uses them.**

| # | Set | Purpose |
|---|---|---|
| 1 | training-side `test` | in-distribution generalisation; identical rows to the TinyBERT baseline |
| 2 | `val_original` (40,909) | untouched AI4Privacy English validation |
| 3 | `val_supplementary` (20%) | weak-label validation, grouped holdout |
| 4 | combined validation | 2 + 3, the model-selection metric |
| 5 | OOD dev | error analysis, free to iterate on |
| 6 | frozen OOD final | run **once**, at the very end, behind `RUN_FROZEN_FINAL` |

**Restartability.** Every expensive stage writes an artifact and is skipped when
that artifact already matches the configuration. Set `FORCE_REBUILD_DATA = True`
to rebuild the corpora from scratch.

**What is committed.** Corpora, checkpoints, weights and ONNX graphs stay local
and gitignored. `config/`, `metrics/` and `tokenizer/` are the tracked evidence,
and `model_v2_data_manifest.json` is what lets a teammate regenerate an
identical supplementary holdout.

## 1. Environment

Verify the runtime before anything expensive happens. `PYTHONUTF8=1` matters on
Windows: without it the modern ONNX exporter dies while printing a progress
marker and the legacy fallback produces `If` subgraphs that cannot be quantized.

In [ ]:
import os, platform, sys

os.environ.setdefault("PYTHONUTF8", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "0")   # set to "1" to forbid downloads

import json, math, random, time, hashlib
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "FullPipeline_V3").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "FullPipeline_V3").is_dir():
    raise RuntimeError("Run this notebook from inside the OnKith repository.")

for extra in (REPO_ROOT / "model" / "privacy_data_v2", REPO_ROOT / "model" / "privacy_evaluation"):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

import transformers

print("repo root        :", REPO_ROOT)
print("python           :", platform.python_version(), "|", platform.platform())
print("torch            :", torch.__version__, "| cuda:", torch.cuda.is_available())
print("transformers     :", transformers.__version__)
try:
    import onnxruntime
    print("onnxruntime      :", onnxruntime.__version__)
except ImportError:
    print("onnxruntime      : not installed (needed from section 24 onwards)")
print("PYTHONUTF8       :", os.environ.get("PYTHONUTF8"))
if torch.cuda.is_available():
    print("gpu              :", torch.cuda.get_device_name(0))

## 2. Configuration

Everything tunable lives in this one cell. Seeds, paths, quotas, split factors,
training hyper-parameters and the frozen-final gate.

In [ ]:
SEED = 42

ARTIFACTS = REPO_ROOT / "model" / "DeBERTa-v3-xsmall" / "artifacts" / "model_v2"
CONFIG_DIR = REPO_ROOT / "model" / "DeBERTa-v3-xsmall" / "config"
DATA_DIR      = ARTIFACTS / "data"          # gitignored
CHECKPOINTS   = ARTIFACTS / "checkpoints"   # gitignored
HF_MODEL_DIR  = ARTIFACTS / "huggingface_model"  # gitignored
ONNX_DIR      = ARTIFACTS / "onnx_model"    # gitignored
METRICS_DIR   = ARTIFACTS / "metrics"       # committed
TOKENIZER_DIR = ARTIFACTS / "tokenizer"     # committed
for directory in (DATA_DIR, CHECKPOINTS, HF_MODEL_DIR, ONNX_DIR, METRICS_DIR, TOKENIZER_DIR, CONFIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_PATH  = DATA_DIR / "model_v2_train.jsonl"
TEST_PATH   = DATA_DIR / "model_v2_test.jsonl"
VAL_SUPP_PATH = DATA_DIR / "model_v2_val_supplementary.jsonl"
VAL_ORIG_PATH = DATA_DIR / "model_v2_val_original.jsonl"
MANIFEST_PATH = CONFIG_DIR / "model_v2_data_manifest.json"
LABEL_MAP_PATH = CONFIG_DIR / "model_v2_label_map.json"
MAPPING_CSV_PATH = CONFIG_DIR / "model_v2_source_mapping.csv"
TRAINING_CONFIG_PATH = CONFIG_DIR / "model_v2_training_config.json"

# --- data build -------------------------------------------------------------
FORCE_REBUILD_DATA = False        # True re-reads every source and re-splits
SAUDI_CSV = REPO_ROOT / "model" / "data" / "saudi_mena_pii_dataset.csv"
HF_TOKEN = os.environ.get("HF_TOKEN") or None
HF_CACHE = os.environ.get("PII_CACHE_DIR") or str(REPO_ROOT / "model" / "data" / ".pii_hf_cache")
# Cap rows read per upstream split. None = read everything (hours).
# Set a small number for a structural smoke run.
MAX_SOURCE_ROWS = None
SUPPLEMENTARY_SOURCES = ["gretel", "nemotron", "kiji", "saudi_mena"]

# --- model ------------------------------------------------------------------
BASE_MODEL = "microsoft/deberta-v3-xsmall"
BASE_MODEL_REVISION = "4b419818330868dff6a60ad3e6b1c730f8b8c0c6"
CONTEXT_LIMIT = 512      # production windowing contract
OVERFLOW_STRIDE = 128

# --- training ---------------------------------------------------------------
MAX_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 2
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 3e-5     # below TinyBERT's 5e-5; deberta-v3 is a stronger backbone
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06      # deberta-v3 is warmup-sensitive
MAX_GRAD_NORM = 1.0
USE_AMP = torch.cuda.is_available()
# Per-epoch validation is subsampled for speed; the selected checkpoint is
# always re-scored on the full sets in sections 16-19.
EPOCH_EVAL_SAMPLE = 6000

# --- evaluation -------------------------------------------------------------
# The frozen OOD final set may be used exactly once, on the selected model.
# Leave this False until every other number has been reviewed.
RUN_FROZEN_FINAL = False

# --- quantization gate ------------------------------------------------------
INT8_MAX_TYPED_F1_DROP = 0.02
INT8_MAX_LEAKAGE_RISE = 0.015

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("artifacts:", ARTIFACTS)
print("committed config:", CONFIG_DIR)
print("RUN_FROZEN_FINAL:", RUN_FROZEN_FINAL, "| FORCE_REBUILD_DATA:", FORCE_REBUILD_DATA)

## 3. Canonical ontology

`privacy_data_v2/ontology.py` is the single source of truth and self-checks at
import. This cell writes the committed label map that the model config, the
ONNX export and every evaluator read back.

In [ ]:
import ontology
from ontology import BIO_LABELS, ENTITIES, ID2LABEL, LABEL2ID, NUM_LABELS, SHARED_WITH_TINYBERT

assert len(ENTITIES) == 31 and NUM_LABELS == 63, (len(ENTITIES), NUM_LABELS)

payload = ontology.label_map_payload()
LABEL_MAP_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print(f"{len(ENTITIES)} entities / {NUM_LABELS} BIO labels")
print("removed :", ", ".join(sorted(ontology.REMOVED_ENTITIES)))
print("added   :", ", ".join(sorted(ontology.ADDED_ENTITIES)))
print("shared with TinyBERT:", len(SHARED_WITH_TINYBERT), "entities")
print("volume-weak labels  :", ", ".join(ontology.WEAK_LABELS))
print("OOD-weak labels     :", ", ".join(sorted(ontology.OOD_WEAK_LABELS)))
print("\nwrote", LABEL_MAP_PATH.relative_to(REPO_ROOT))

## 4. Source registry and label mapping audit

Every source is declared with its licence and split names. Every native label
gets a decision with a written reason; nothing is ever guessed. This writes the
committed mapping CSV that a reviewer can read without running anything.

In [ ]:
import label_mapping, sources
from label_mapping import mapping_decision

mapping_summary = label_mapping.write_mapping_csv(MAPPING_CSV_PATH)
print(json.dumps(mapping_summary, indent=2))

rows = label_mapping.mapping_table()
mapped = [row for row in rows if row["status"] == "mapped"]
targets = {row["onkith_label"] for row in mapped}
assert not targets - set(ENTITIES), f"mapping targets outside the ontology: {targets - set(ENTITIES)}"
print("\nontology labels with no source mapping:", sorted(set(ENTITIES) - targets) or "none")

print("\nmedium-confidence mappings needing review:")
for row in mapped:
    if row["confidence"] == "medium":
        gate = f"  [gate: {row['requires_span_gate']}]" if row["requires_span_gate"] else ""
        print(f"  {row['source_dataset']:12} {row['native_label']:28} -> {row['onkith_label']}{gate}")

print("\nrejected concepts (sample):")
for row in [r for r in rows if r["status"] != "mapped"][:12]:
    print(f"  {row['native_label']:30} {row['reason']}")

## 5. AI4Privacy base dataset

The pinned English source, split exactly as TinyBERT split it, so the
training-side `test` set is the same 16,374 rows the frozen baseline used.

* official English train **163,740** -> project train **147,366** + project
  validation **16,374** (the latter becomes Model V2 `test`);
* official English validation **40,909** -> `val_original`, untouched.

`HEIGHT` (one span, validation only) and every other removed label are dropped
to `O`; the row itself is kept, so `val_original` keeps all 40,909 rows.

In [ ]:
import data_audit
from records import make_record

def ai4privacy_records(dataset_split, indices, partition):
    """Convert a pinned AI4Privacy split into canonical Model V2 records."""
    supported = set(ENTITIES)
    out, dropped = [], Counter()
    subset = dataset_split.select(indices) if indices is not None else dataset_split
    for row in subset:
        text = str(row["source_text"])
        spans = []
        for span in row["privacy_mask"]:
            label = str(span["label"])
            if label not in supported:
                dropped[label] += 1
                continue
            start, end = int(span["start"]), int(span["end"])
            if 0 <= start < end <= len(text) and text[start:end] == str(span["value"]):
                spans.append({"label": label, "start": start, "end": end, "value": text[start:end]})
            else:
                dropped[f"{label}:invalid_offset"] += 1
        spans.sort(key=lambda s: (s["start"], s["end"]))
        deduped, last_end = [], -1
        for span in spans:
            if span["start"] >= last_end:
                deduped.append(span); last_end = span["end"]
            else:
                dropped[f"{span['label']}:overlap"] += 1
        out.append(make_record(
            uid=f"ai4privacy:{row.get('uid', len(out))}",
            source_text=text, privacy_mask=deduped,
            source_dataset="ai4privacy", partition=partition,
            source_split=partition, region=str(row.get("region") or "") or None,
        ))
    return out, dropped


if FORCE_REBUILD_DATA or not VAL_ORIG_PATH.exists():
    from datasets import load_dataset
    os.environ.setdefault("HF_HOME", HF_CACHE)
    dataset = load_dataset(sources.AI4PRIVACY["name"], revision=sources.AI4PRIVACY["revision"])

    english_train = dataset["train"].filter(lambda row: row["language"] == "en")
    english_validation = dataset["validation"].filter(lambda row: row["language"] == "en")
    print("official English train     :", len(english_train))
    print("official English validation:", len(english_validation))
    assert len(english_validation) == 40_909, "val_original must be the full 40,909 rows"

    labels_by_row, _, indicator = data_audit.extract_row_entity_types(english_train["privacy_mask"])
    train_idx, test_idx = data_audit.historical_multilabel_split(
        indicator, labels_by_row, validation_fraction=0.10, seed=42
    )
    print("project train / project validation:", len(train_idx), "/", len(test_idx))

    ai4_train, dropped_train = ai4privacy_records(english_train, train_idx, "train")
    ai4_test, _ = ai4privacy_records(english_train, test_idx, "test")
    val_original, dropped_val = ai4privacy_records(english_validation, None, "val_original")

    import records as records_module
    records_module.write_jsonl(VAL_ORIG_PATH, val_original)
    records_module.write_jsonl(DATA_DIR / "ai4privacy_train.jsonl", ai4_train)
    records_module.write_jsonl(DATA_DIR / "ai4privacy_test.jsonl", ai4_test)
    print("\nspans dropped to O (train):", dict(dropped_train.most_common(8)))
    print("spans dropped to O (val)  :", dict(dropped_val.most_common(8)))
else:
    import records as records_module
    ai4_train = records_module.load_jsonl(DATA_DIR / "ai4privacy_train.jsonl", validate=False)
    ai4_test = records_module.load_jsonl(DATA_DIR / "ai4privacy_test.jsonl", validate=False)
    val_original = records_module.load_jsonl(VAL_ORIG_PATH, validate=False)
    print("reused cached AI4Privacy artifacts")

print(f"\nai4_train={len(ai4_train):,}  ai4_test={len(ai4_test):,}  val_original={len(val_original):,}")
assert len(val_original) == 40_909

## 6. Contamination firewall

Built from every evaluation set **before** a single training row is written. Any
candidate whose normalised text already exists in an evaluation set is rejected
outright. The frozen OOD-final fixture is loaded here for fingerprints only -
its content is never read into training material and never scored until
section 23.

In [ ]:
import pools
from records import from_fixture_case
from privacy_eval.schema import load_cases

FIXTURES = REPO_ROOT / "model" / "privacy_evaluation" / "fixtures"
ood_dev = [from_fixture_case(case.to_dict(), partition="ood_dev")
           for case in load_cases(FIXTURES / "ood_dev.jsonl")]
ood_final = [from_fixture_case(case.to_dict(), partition="ood_final")
             for case in load_cases(FIXTURES / "ood_final.jsonl")]

firewall = pools.build_firewall(
    val_original=val_original, ood_dev=ood_dev, ood_final=ood_final
)
print(f"firewall holds {firewall.size:,} protected fingerprints")
print(f"  val_original {len(val_original):,} | ood_dev {len(ood_dev)} | ood_final {len(ood_final)}")
print("\nood_final is fingerprinted only. It is not scored until section 23,")
print("and RUN_FROZEN_FINAL is currently", RUN_FROZEN_FINAL)

## 7. Supplementary selection

Gretel, Nemotron, Kiji and the Saudi generator are read row by row. Each row is
mapped into the 63-label ontology, its `PASSWORD` spans are gated by
`password_rules`, and it is kept only if it strengthens a weak label - by volume
for labels the base cannot teach at all, by novelty for labels that have volume
but score badly out of distribution.

Selected rows keep **every** correctly mapped span, not just the triggering one.

If Nemotron is gated or its schema has drifted, it is skipped and the failure is
recorded; the build continues on the remaining sources.

In [ ]:
import selection, dedup as dedup_module
from selection import SelectionConfig, SupplementarySelector

if FORCE_REBUILD_DATA or not VAL_SUPP_PATH.exists():
    selector = SupplementarySelector(SelectionConfig())
    deduplicator = dedup_module.Deduplicator()
    mapping_audit = Counter()
    source_status = {}
    candidates = []

    for source in SUPPLEMENTARY_SOURCES:
        spec = sources.SUPPLEMENTARY_SOURCES[source]
        kept_here = 0
        try:
            if source == "saudi_mena":
                if not SAUDI_CSV.exists():
                    raise FileNotFoundError(
                        f"{SAUDI_CSV} is missing. Run model/data/saudi_PII_Builder.ipynb first."
                    )
                streams = [("generated", sources.iter_saudi_rows(str(SAUDI_CSV)))]
            else:
                streams = [
                    (split, sources.iter_source_rows(
                        source, split, hf_token=HF_TOKEN,
                        cache_dir=HF_CACHE, limit=MAX_SOURCE_ROWS))
                    for split in spec["splits"]
                ]
            for split, stream in streams:
                for raw in stream:
                    record = sources.normalize_row(
                        source=source, row=raw, partition="candidate", audit=mapping_audit
                    )
                    if record is None:
                        continue
                    if not firewall.accept(record["source_text"]):
                        continue
                    kept, outcome = selector.consider(record)
                    if kept is None:
                        continue
                    keep, _ = deduplicator.check_and_add(kept["source_text"], kept["privacy_mask"])
                    if not keep:
                        continue
                    candidates.append(kept)
                    kept_here += 1
            source_status[source] = {"status": "ok", "selected_rows": kept_here,
                                     "revision": sources.resolve_revision(source, hf_token=HF_TOKEN)}
        except Exception as exc:
            source_status[source] = {
                "status": "unavailable", "selected_rows": kept_here,
                "error": f"{type(exc).__name__}: {exc}",
            }
            print(f"  !! {source} unavailable: {type(exc).__name__}: {exc}")
            if not spec.get("optional") and source != "saudi_mena":
                raise

    selection_audit = selector.audit()
    password_audit = __import__("password_rules").audit_password_decisions(selector.password_decisions)
    dedup_audit = deduplicator.audit()
    firewall_audit = firewall.audit()
else:
    candidates = None
    print("cached supplementary split found; section 8 will reuse it")

if candidates is not None:
    print(f"\nselected supplementary rows: {len(candidates):,}")
    print(json.dumps(source_status, indent=2))
    print("\nspans selected by label:")
    for label, count in sorted(selection_audit["selected_spans_by_label"].items()):
        carried = selection_audit["carried_spans_by_label"].get(label, 0)
        target = selection_audit["config"]["span_targets"].get(label)
        note = f" target={target}" if target else ""
        print(f"  {label:18} {count:7,}  (carried {carried:6,}){note}")
    if selection_audit["unmet_targets"]:
        print("\nunmet targets (measured evidence, not a failure to hide):")
        print(json.dumps(selection_audit["unmet_targets"], indent=2))
    print("\nPASSWORD gate:", json.dumps(password_audit, indent=2))
    print("\ndedup:", json.dumps(dedup_audit, indent=2))
    print("contamination firewall:", json.dumps(
        {k: v for k, v in firewall_audit.items() if k != "examples_by_protected_set"}, indent=2))

## 8. Grouped 80/20 supplementary split

Rows are connected when they share **any** of the declared factors - entity
value, email domain, canonical template, formatting family, numeric pattern
family or organisation family - so a near-identical value or template cannot sit
on both sides. The split is then label-aware: every label with enough
independent components contributes its share to the holdout.

The 20% holdout is written to a local, gitignored artifact and is regenerable
byte-for-byte from the committed manifest.

In [ ]:
from grouped_splits import MODEL_V2_SPLIT_FACTORS
import records as records_module

if candidates is not None:
    supp_train, val_supplementary, supp_split_audit = pools.split_supplementary(
        candidates,
        holdout_fraction=pools.SUPPLEMENTARY_HOLDOUT_FRACTION,
        seed=pools.SEED_SUPPLEMENTARY_HOLDOUT,
        factors=MODEL_V2_SPLIT_FACTORS,
    )
    val_supp_artifact = records_module.write_jsonl(VAL_SUPP_PATH, val_supplementary)
    records_module.write_jsonl(DATA_DIR / "supplementary_training_side.jsonl", supp_train)
else:
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f"{MANIFEST_PATH} is missing, so the cached holdout cannot be verified. "
            "Set FORCE_REBUILD_DATA = True and re-run from section 7."
        )
    cached = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    supp_train = records_module.load_jsonl(DATA_DIR / "supplementary_training_side.jsonl", validate=False)
    val_supplementary = records_module.load_jsonl(VAL_SUPP_PATH, validate=False)
    supp_split_audit = cached["split"]["supplementary"]
    val_supp_artifact = cached["artifacts"]["val_supplementary"]
    selection_audit = cached.get("selection", {})
    password_audit = cached.get("password", {})
    dedup_audit = cached.get("deduplication", {})
    source_status = cached.get("sources", {}).get("status", {})

print(f"supplementary training side: {len(supp_train):,}")
print(f"val_supplementary          : {len(val_supplementary):,}")
print(f"holdout sha256             : {val_supp_artifact['sha256']}")
print("\nfactors:", supp_split_audit["factors"])
print("disjoint on every factor:", supp_split_audit["is_disjoint"])
print("components:", supp_split_audit.get("component_count"),
      "| largest:", supp_split_audit.get("largest_component_rows"),
      f"({supp_split_audit.get('largest_component_fraction', 0):.2%} of the pool)")
if supp_split_audit.get("largest_component_fraction", 0) > 0.05:
    print("  WARNING: one component holds >5% of the pool; review before trusting the split.")
assert supp_split_audit["is_disjoint"], "a declared factor crossed the split boundary"

thin = supp_split_audit.get("labels_without_independent_groups", [])
if thin:
    print("\nlabels without enough independent groups to validate this cycle:")
    print(" ", ", ".join(thin))
    print("  These are reported, not forced. Per-label claims for them are not supported.")

## 9. Training-side pool and train/test split

The AI4Privacy component keeps its pinned historical partition, so the
training-side `test` set stays directly comparable with the frozen TinyBERT
baseline. The supplementary training side is split by the same grouped
mechanism.

In [ ]:
train_rows, test_rows, training_split_audit = pools.split_training_side(
    ai4_train, ai4_test, supp_train,
    test_fraction=pools.SUPPLEMENTARY_TEST_FRACTION,
    seed=pools.SEED_SUPPLEMENTARY_TEST,
    factors=MODEL_V2_SPLIT_FACTORS,
)
print(json.dumps({k: v for k, v in training_split_audit.items()
                  if k in {"ai4privacy_train_rows", "ai4privacy_test_rows",
                           "supplementary_train_rows", "supplementary_test_rows",
                           "total_train_rows", "total_test_rows", "is_disjoint"}}, indent=2))

firewall.assert_clean(train_rows, label="train")
firewall.assert_clean(test_rows, label="test")
print("\nfirewall: no train/test row appears in val_original, ood_dev or ood_final")

## 10. Training-only augmentation

One serious training run: real supplementary data **and** conservative,
span-safe augmentation together. Only field-proven transforms are used, each
rebuilding the text and recalculating offsets. No paraphrasing, no LLM
rewriting, no synonym substitution, no backtranslation - all of those can move
an entity boundary silently.

Two additions target measured failures:

* **PASSWORD disclosures**, generated with exact spans and re-checked through
  the same admission gate applied to external corpora;
* **hard negatives**, because AI4Privacy English contains only 44 no-PII rows in
  163,740 and the frozen baseline over-masks 65.28% of no-PII cases.

Caps keep this honest: at most 25% of the split augmented, at most 8% from any
one family, and every family must draw on at least 50 distinct parent rows.
Nothing here touches `test`, `val_original`, `val_supplementary` or either OOD
set - the guard in `augmentation._assert_mutable` rejects them outright.

In [ ]:
import augment_pipeline
from augment_pipeline import AugmentationPlan, augment_training_rows

resources = augment_pipeline.load_resources()

# Entity values that appear in the protected holdout must not be reused as
# augmentation material, or the grouped split would leak through the back door.
holdout_values = defaultdict(set)
for record in val_supplementary + val_original:
    for span in record["privacy_mask"]:
        holdout_values[str(span["label"])].add(str(span["value"]))
resources = dict(resources)
resources["entity_values"] = augment_pipeline.restrict_inventories(
    resources["entity_values"], holdout_values
)
print("inventory sizes after holdout filtering:")
for label in ontology.WEAK_LABELS:
    print(f"  {label:18} {len(resources['entity_values'].get(label, []))}")

# --- generated additions ----------------------------------------------------
hard_negatives = augment_pipeline.generate_model_v2_hard_negatives(seed=SEED)
target_hard_negatives = int(len(train_rows) * augment_pipeline.HARD_NEGATIVE_SHARE)
password_rows = augment_pipeline.generate_password_disclosures(
    resources["entity_values"], count=2000, seed=SEED
)
print(f"\ncurated hard-negative seeds : {len(hard_negatives)}")
print(f"hard-negative target rows   : {target_hard_negatives:,} (reached via span-safe variants below)")
print(f"password disclosure rows    : {len(password_rows):,}")

base_train = train_rows + hard_negatives + password_rows

plan = AugmentationPlan(base_rows=len(base_train), seed=SEED)
augmentation = augment_training_rows(base_train, resources, plan)
augmented_train = base_train + augmentation.rows
augmentation_audit = augmentation.audit(plan)

print("\n" + json.dumps({k: v for k, v in augmentation_audit.items() if k != "skipped_by_reason"}, indent=2))
if not augmentation_audit["parent_floor_satisfied"]:
    print("\nWARNING: these families drew on too few parents and are oversampling:")
    print(" ", ", ".join(augmentation_audit["families_below_parent_floor"]))

train_rows = augmented_train
print(f"\nfinal train rows: {len(train_rows):,}  (base {len(base_train):,} + augmented {len(augmentation.rows):,})")
empty = sum(1 for record in train_rows if not record["privacy_mask"])
print(f"no-PII rows in train: {empty:,} ({empty / len(train_rows):.2%})")

## 11. Statistics, leakage proof and the reproducibility manifest

The assertions here are the reason the generated corpora can stay out of Git:
they prove, rather than assume, that no evaluation row is a training row. The
manifest records everything a teammate needs to regenerate an identical
supplementary holdout.

In [ ]:
import manifest as manifest_module

records_module.write_jsonl(TRAIN_PATH, train_rows)
records_module.write_jsonl(TEST_PATH, test_rows)

partitions = {
    "train": train_rows,
    "test": test_rows,
    "val_original": val_original,
    "val_supplementary": val_supplementary,
    "ood_dev": ood_dev,
    "ood_final": ood_final,
}
disjointness = pools.assert_partitions_disjoint(partitions)
print("every partition is text-disjoint from every other partition")
print(json.dumps(disjointness["partitions"], indent=2))

statistics = {name: pools.partition_statistics(rows) for name, rows in partitions.items()}
print("\nspans by label (train vs the two validation sets):")
print(f"  {'label':18} {'train':>10} {'val_orig':>10} {'val_supp':>10}")
for label in ENTITIES:
    print(f"  {label:18} "
          f"{statistics['train']['spans_by_label'].get(label, 0):>10,} "
          f"{statistics['val_original']['spans_by_label'].get(label, 0):>10,} "
          f"{statistics['val_supplementary']['spans_by_label'].get(label, 0):>10,}")
print(f"\ntrain email domains: {statistics['train']['unique_email_domains']} "
      f"(the pinned AI4Privacy training distribution had only 7)")

artifacts_record = {
    "train": records_module.write_jsonl(TRAIN_PATH, train_rows),
    "test": records_module.write_jsonl(TEST_PATH, test_rows),
    "val_supplementary": val_supp_artifact,
    "val_original": {"path": str(VAL_ORIG_PATH), "rows": len(val_original),
                     "note": "Reconstructible from the pinned upstream revision; not required in Git."},
}

source_payload = sources.source_registry_payload(SUPPLEMENTARY_SOURCES)
source_payload["status"] = source_status

data_manifest = manifest_module.build_manifest(
    sources=source_payload,
    mapping_summary=mapping_summary,
    selection_audit=selection_audit,
    password_audit=password_audit,
    dedup_audit=dedup_audit,
    firewall_audit=firewall.audit(),
    supplementary_split_audit=supp_split_audit,
    training_split_audit=training_split_audit,
    augmentation_audit=augmentation_audit,
    partition_statistics=statistics,
    disjointness=disjointness,
    artifacts=artifacts_record,
    config={
        "seed": SEED, "base_model": BASE_MODEL, "base_model_revision": BASE_MODEL_REVISION,
        "context_limit": CONTEXT_LIMIT, "overflow_stride": OVERFLOW_STRIDE,
        "supplementary_sources": SUPPLEMENTARY_SOURCES,
        "max_source_rows": MAX_SOURCE_ROWS,
        "split_factors": list(MODEL_V2_SPLIT_FACTORS),
        "supplementary_holdout_fraction": pools.SUPPLEMENTARY_HOLDOUT_FRACTION,
        "supplementary_test_fraction": pools.SUPPLEMENTARY_TEST_FRACTION,
    },
)
manifest_module.write_manifest(MANIFEST_PATH, data_manifest)
print("\nwrote", MANIFEST_PATH.relative_to(REPO_ROOT))
print("regenerate the holdout by re-running sections 1-11; verify with")
print("  privacy_data_v2.manifest.verify_manifest(committed, regenerated)")

## 12. Tokenization and BIO alignment

The 512/128 windowing contract matches production exactly. Labels are aligned
from character spans onto token offsets; special tokens and sub-word
continuations receive `-100` so they are ignored by the loss.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_MODEL_REVISION, use_fast=True)
assert tokenizer.is_fast, "a fast tokenizer is required for exact character offsets"


def encode_records(rows, *, desc=""):
    """Window and align one partition. Returns a list of feature dicts."""
    features = []
    for index, record in enumerate(rows):
        text = record["source_text"]
        encoded = tokenizer(
            text, truncation=True, max_length=CONTEXT_LIMIT, stride=OVERFLOW_STRIDE,
            return_overflowing_tokens=True, return_offsets_mapping=True,
            return_attention_mask=True, padding=False, return_token_type_ids=False,
        )
        spans = record["privacy_mask"]
        for window in range(len(encoded["input_ids"])):
            offsets = encoded["offset_mapping"][window]
            labels = []
            for start, end in offsets:
                if start == end:
                    labels.append(-100); continue
                tag = "O"
                for span in spans:
                    if start >= int(span["start"]) and end <= int(span["end"]):
                        prefix = "B" if start == int(span["start"]) else "I"
                        tag = f"{prefix}-{span['label']}"
                        break
                    if start < int(span["end"]) and end > int(span["start"]):
                        # Partial overlap: the token straddles a boundary. Tag it
                        # with the entity so the character is never left as O.
                        prefix = "B" if start <= int(span["start"]) else "I"
                        tag = f"{prefix}-{span['label']}"
                        break
                labels.append(LABEL2ID[tag])
            features.append({
                "input_ids": encoded["input_ids"][window],
                "attention_mask": encoded["attention_mask"][window],
                "labels": labels,
                "offset_mapping": offsets,
                "row_index": index,
            })
        if desc and index and index % 25_000 == 0:
            print(f"  {desc}: {index:,}/{len(rows):,}")
    return features


train_features = encode_records(train_rows, desc="train")
print(f"train windows: {len(train_features):,} from {len(train_rows):,} rows")
lengths = np.array([len(f["input_ids"]) for f in train_features])
print("token length p50/p90/p95/p99/max:",
      *[int(np.percentile(lengths, p)) for p in (50, 90, 95, 99)], int(lengths.max()))

### 12b. Alignment sanity checks

These must pass before training. A silent alignment bug produces a model that
looks fine on token accuracy and fails completely on spans.

In [ ]:
def check_alignment(rows, features, sample=400):
    """Every gold span must be covered by at least one window, with the right tag."""
    by_row = defaultdict(list)
    for feature in features:
        by_row[feature["row_index"]].append(feature)

    checked = uncovered = mismatched = 0
    rng = random.Random(SEED)
    indices = rng.sample(range(len(rows)), min(sample, len(rows)))
    for index in indices:
        record = rows[index]
        covered_characters = set()
        for feature in by_row[index]:
            for (start, end), label_id in zip(feature["offset_mapping"], feature["labels"]):
                if label_id in (-100, LABEL2ID["O"]) or start == end:
                    continue
                label = ID2LABEL[label_id]
                entity = label[2:]
                matching = [s for s in record["privacy_mask"]
                            if str(s["label"]) == entity
                            and start < int(s["end"]) and end > int(s["start"])]
                if not matching:
                    mismatched += 1
                else:
                    covered_characters.update(range(start, end))
        for span in record["privacy_mask"]:
            checked += 1
            wanted = set(range(int(span["start"]), int(span["end"])))
            if not wanted & covered_characters:
                uncovered += 1
    return {"spans_checked": checked, "uncovered_spans": uncovered, "mismatched_tokens": mismatched}


alignment = check_alignment(train_rows, train_features)
print(json.dumps(alignment, indent=2))
assert alignment["uncovered_spans"] == 0, "a gold span was not covered by any window"
assert alignment["mismatched_tokens"] == 0, "a token was tagged with the wrong entity"

# No orphan I- tags, and every label id is inside the 63-label range.
for feature in train_features[:5000]:
    previous = None
    for label_id in feature["labels"]:
        assert label_id == -100 or 0 <= label_id < NUM_LABELS
        if label_id == -100:
            continue
        label = ID2LABEL[label_id]
        if label.startswith("I-"):
            assert previous is not None and previous[2:] == label[2:], f"orphan {label}"
        previous = None if label == "O" else label
print("no orphan I- tags; all label ids inside [0, 62]")

example = train_features[0]
print("\nsample window:")
for token, (start, end), label_id in list(zip(
        tokenizer.convert_ids_to_tokens(example["input_ids"]),
        example["offset_mapping"], example["labels"]))[:24]:
    tag = "IGN" if label_id == -100 else ID2LABEL[label_id]
    print(f"  {token:<18} {str((start, end)):<12} {tag}")

## 13. Load DeBERTa-v3-xsmall with the 63-label head

The preflight established the deployment contract at 67 labels with a random
head. Only the head width changes here (25,795 -> 24,255 parameters, about
0.002% of the model), so every tokenizer, ONNX, quantization and runtime result
from the preflight still applies.

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL, revision=BASE_MODEL_REVISION,
    num_labels=NUM_LABELS,
    id2label={index: label for index, label in ID2LABEL.items()},
    label2id=dict(LABEL2ID),
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

total = sum(p.numel() for p in model.parameters())
head = sum(p.numel() for p in model.classifier.parameters())
print(f"{BASE_MODEL} @ {BASE_MODEL_REVISION[:12]}")
print(f"parameters: {total:,} (backbone {total - head:,} + head {head:,})")
print(f"labels    : {model.config.num_labels}")
print(f"device    : {device}")
assert model.config.num_labels == 63

## 14. Batching

Dynamic padding to the longest window in each batch, rounded to a multiple of 8.
Labels pad with `-100`. The training-side percentiles from section 12 show p50
around 75 tokens, so padding to the batch maximum costs far less than padding to
512.

In [ ]:
from torch.utils.data import DataLoader, Dataset


class WindowDataset(Dataset):
    def __init__(self, features):
        self.features = features

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        feature = self.features[index]
        return {
            "input_ids": feature["input_ids"],
            "attention_mask": feature["attention_mask"],
            "labels": feature["labels"],
        }


def collate(batch):
    longest = max(len(item["input_ids"]) for item in batch)
    longest = int(math.ceil(longest / 8) * 8)
    pad_id = tokenizer.pad_token_id
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for item in batch:
        padding = longest - len(item["input_ids"])
        out["input_ids"].append(item["input_ids"] + [pad_id] * padding)
        out["attention_mask"].append(item["attention_mask"] + [0] * padding)
        out["labels"].append(item["labels"] + [-100] * padding)
    return {key: torch.tensor(value, dtype=torch.long) for key, value in out.items()}


generator = torch.Generator(); generator.manual_seed(SEED)
train_loader = DataLoader(
    WindowDataset(train_features), batch_size=TRAIN_BATCH_SIZE, shuffle=True,
    collate_fn=collate, generator=generator, num_workers=0,
)
print(f"train batches per epoch: {len(train_loader):,}")

## 15. Span prediction

Decoding reuses the **production** functions - `consolidate_predictions_to_words`
for whole-word promotion and the same BIO repair - so an evaluation number here
reflects what the deployed pipeline would actually emit, not a notebook-only
decoding path. The structured regex layer is deliberately excluded: these
numbers measure what the neural model learned.

In [ ]:
sys.path.insert(0, str(REPO_ROOT / "FullPipeline_V3"))
from integrated_pipeline.span_merger import consolidate_predictions_to_words
from privacy_eval.model_v2_adapter import repair_bio


@torch.no_grad()
def predict_spans(rows, *, batch_size=EVAL_BATCH_SIZE, torch_model=None):
    """Predict character spans for canonical records, production-style."""
    active = torch_model if torch_model is not None else model
    active.eval()
    features = encode_records(rows)
    by_row = defaultdict(list)
    for feature in features:
        by_row[feature["row_index"]].append(feature)

    logits_by_feature = {}
    order = sorted(range(len(features)), key=lambda i: len(features[i]["input_ids"]))
    for start in range(0, len(order), batch_size):
        chunk = [features[i] for i in order[start:start + batch_size]]
        batch = collate([
            {"input_ids": f["input_ids"], "attention_mask": f["attention_mask"], "labels": f["labels"]}
            for f in chunk
        ])
        batch = {key: value.to(device) for key, value in batch.items() if key != "labels"}
        with torch.autocast("cuda", enabled=USE_AMP):
            output = active(**batch).logits.float()
        probabilities = torch.softmax(output, dim=-1)
        confidence, prediction = probabilities.max(dim=-1)
        for position, feature in enumerate(chunk):
            length = len(feature["input_ids"])
            logits_by_feature[id(feature)] = (
                prediction[position, :length].cpu().numpy(),
                confidence[position, :length].cpu().numpy(),
            )

    predictions = []
    for index, record in enumerate(rows):
        text = record["source_text"]
        best = {}
        for feature in by_row[index]:
            prediction, confidence = logits_by_feature[id(feature)]
            for (start, end), label_id, score in zip(
                    feature["offset_mapping"], prediction, confidence):
                if start == end:
                    continue
                key = (int(start), int(end))
                if key not in best or float(score) > best[key][1]:
                    best[key] = (int(label_id), float(score))
        offsets = sorted(best)
        if not offsets:
            predictions.append([]); continue
        labels, _ = repair_bio([ID2LABEL[best[o][0]] for o in offsets])
        scores = [best[o][1] for o in offsets]
        consolidated = consolidate_predictions_to_words(text, offsets, labels, scores)
        predictions.append([
            {"label": span.entity_type, "start": span.start_char, "end": span.end_char}
            for span in consolidated.spans
        ])
    return predictions

## 16. Train

One complete run. AdamW with linear warmup over the first 6% of steps then
linear decay - DeBERTa-v3 is warmup-sensitive and destabilises without it. Up to
10 epochs with early stopping after 2 epochs without improvement.

**Model selection** uses typed entity F1 on the *combined* validation view
(`val_original` + `val_supplementary`), subsampled per epoch for speed. Selecting
on `val_original` alone would reward IID overfitting; selecting on
`val_supplementary` alone would reward the gap-filler distribution.

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from privacy_eval.dataset_metrics import evaluate_spans

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * MAX_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(total_steps * WARMUP_RATIO), total_steps
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

rng = random.Random(SEED)
epoch_val = (
    rng.sample(val_original, min(EPOCH_EVAL_SAMPLE // 2, len(val_original)))
    + rng.sample(val_supplementary, min(EPOCH_EVAL_SAMPLE // 2, len(val_supplementary)))
)
print(f"per-epoch validation sample: {len(epoch_val):,} rows "
      f"(full sets are scored in sections 18-21)")

history, best_metric, best_epoch, patience_left = [], -1.0, -1, EARLY_STOPPING_PATIENCE
BEST_CHECKPOINT = CHECKPOINTS / "best.pt"

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running, started = 0.0, time.perf_counter()
    for step, batch in enumerate(train_loader, start=1):
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", enabled=USE_AMP):
            loss = model(**batch).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item()
        if step % 500 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_loader)} loss {running / step:.4f}")

    metrics = evaluate_spans(epoch_val, predict_spans(epoch_val))
    typed_f1 = metrics["typed_entity"]["f1"]
    entry = {
        "epoch": epoch,
        "train_loss": running / len(train_loader),
        "combined_typed_f1": typed_f1,
        "combined_char_recall": metrics["private_character_recall"],
        "combined_leakage": metrics["private_character_leakage"],
        "hard_negative_fp_rate": metrics["hard_negative_false_positive_rate"],
        "seconds": time.perf_counter() - started,
        "learning_rate": scheduler.get_last_lr()[0],
    }
    history.append(entry)
    print(f"epoch {epoch}: loss {entry['train_loss']:.4f} | typed F1 {typed_f1:.4f} | "
          f"leakage {entry['combined_leakage']:.4f} | hard-neg FP {entry['hard_negative_fp_rate']:.4f} | "
          f"{entry['seconds']:.0f}s")

    if typed_f1 > best_metric:
        best_metric, best_epoch, patience_left = typed_f1, epoch, EARLY_STOPPING_PATIENCE
        torch.save({"epoch": epoch, "model_state": model.state_dict(),
                    "typed_f1": typed_f1}, BEST_CHECKPOINT)
        print(f"  new best; checkpoint saved")
    else:
        patience_left -= 1
        print(f"  no improvement; patience left {patience_left}")
        if patience_left <= 0:
            print(f"early stopping after epoch {epoch}")
            break

model.load_state_dict(torch.load(BEST_CHECKPOINT, map_location=device)["model_state"])
model.to(device)
print(f"\nselected epoch {best_epoch} with combined typed F1 {best_metric:.4f}")
(METRICS_DIR / "training_history.json").write_text(
    json.dumps({"history": history, "best_epoch": best_epoch,
                "best_combined_typed_f1": best_metric,
                "selection_metric": "typed entity F1 on combined validation (subsampled)"},
               indent=2) + "\n", encoding="utf-8")

## 17. Evaluate the six sets

Each set is scored separately and kept separately. Sets 1-4 run now; OOD dev is
section 21 and the frozen final set is section 22.

In [ ]:
from privacy_eval.dataset_metrics import evaluate_spans, password_analysis, restrict_to_labels, subset_by

evaluations = {}


def evaluate_partition(name, rows, *, extras=True):
    started = time.perf_counter()
    predicted = predict_spans(rows)
    metrics = evaluate_spans(rows, predicted)
    metrics["evaluation_seconds"] = time.perf_counter() - started
    if extras:
        metrics["by_source"] = subset_by(rows, predicted, "source_dataset")
        metrics["password"] = password_analysis(rows, predicted)
    evaluations[name] = metrics
    print(f"{name:22} rows {metrics['rows']:>7,} | typed F1 {metrics['typed_entity']['f1']:.4f} "
          f"| char recall {metrics['private_character_recall']:.4f} "
          f"| leakage {metrics['private_character_leakage']:.4f} "
          f"| hard-neg FP {metrics['hard_negative_false_positive_rate']:.4f}")
    return metrics, predicted


print("=== 1. training-side test (same rows as the TinyBERT baseline) ===")
test_metrics, test_predictions = evaluate_partition("test", test_rows)

print("\n=== 2. val_original (40,909 untouched AI4Privacy English validation) ===")
val_orig_metrics, val_orig_predictions = evaluate_partition("val_original", val_original)

print("\n=== 3. val_supplementary (grouped 20% holdout) ===")
val_supp_metrics, val_supp_predictions = evaluate_partition("val_supplementary", val_supplementary)

print("\n=== 4. combined Model V2 validation ===")
combined_rows = val_original + val_supplementary
combined_predictions = val_orig_predictions + val_supp_predictions
combined_metrics = evaluate_spans(combined_rows, combined_predictions)
combined_metrics["composition"] = {
    "val_original_rows": len(val_original),
    "val_supplementary_rows": len(val_supplementary),
    "note": "Reconstructed from the two sets above; never materialised as a third file.",
}
evaluations["combined_validation"] = combined_metrics
print(f"{'combined_validation':22} rows {combined_metrics['rows']:>7,} | "
      f"typed F1 {combined_metrics['typed_entity']['f1']:.4f} | "
      f"char recall {combined_metrics['private_character_recall']:.4f} | "
      f"leakage {combined_metrics['private_character_leakage']:.4f}")

## 18. Per-label and error analysis

Per-label numbers are reported against the set that can actually support them:
`val_original` for labels the base distribution teaches, `val_supplementary` for
the weak labels the gap-fillers were selected to strengthen.

In [ ]:
print(f"{'label':18} {'val_orig F1':>12} {'supp F1':>10} {'supp sup':>9}  note")
for label in ENTITIES:
    original = val_orig_metrics["per_label"].get(label, {})
    supplementary = val_supp_metrics["per_label"].get(label, {})
    notes = []
    if label in ontology.WEAK_LABELS:
        notes.append("weak")
    if label in ontology.OOD_WEAK_LABELS:
        notes.append(f"ood-weak ({ontology.OOD_WEAK_LABELS[label]:.3f} tinybert)")
    print(f"{label:18} {original.get('f1', float('nan')):>12.4f} "
          f"{supplementary.get('f1', float('nan')):>10.4f} "
          f"{supplementary.get('support', 0):>9,}  {', '.join(notes)}")

print("\nboundary-correct but type-wrong confusions (combined validation, top 12):")
for pair, count in list(combined_metrics["boundary_correct_type_confusions"].items())[:12]:
    print(f"  {pair:36} {count:,}")

print("\nno-PII rows in each evaluation set:")
for name in ("test", "val_original", "val_supplementary"):
    metrics = evaluations[name]
    print(f"  {name:20} {metrics['hard_negative_rows']:>6,} rows | "
          f"FP rate {metrics['hard_negative_false_positive_rate']:.4f}")

## 19. PASSWORD analysis

`PASSWORD` is the one new label, so it gets its own report. A single F1 would
hide the distinction that matters: catching a disclosed credential is only
useful if talking *about* passwords does not trigger it.

In [ ]:
for name in ("test", "val_original", "val_supplementary"):
    analysis = evaluations[name].get("password")
    if not analysis:
        continue
    print(f"--- {name} ---")
    print(f"  credential-disclosure rows      : {analysis['disclosure_rows']:,}")
    if analysis["disclosure_metrics"]:
        disclosure = analysis["disclosure_metrics"]["per_label"].get("PASSWORD", {})
        print(f"  disclosure PASSWORD P/R/F1      : "
              f"{disclosure.get('precision', 0):.4f} / {disclosure.get('recall', 0):.4f} / "
              f"{disclosure.get('f1', 0):.4f}  (support {disclosure.get('support', 0)})")
    print(f"  password-related no-PII rows    : {analysis['password_hard_negative_rows']:,}")
    print(f"  false PASSWORD on those rows    : {analysis['password_hard_negative_false_positives']:,} "
          f"({analysis['password_hard_negative_false_positive_rate']:.4f})")

print("\nA useful PASSWORD label needs high disclosure recall AND a low")
print("hard-negative rate. If the second number is high, the model has learned")
print("the word 'password' rather than the presence of a secret.")

## 20. TinyBERT comparison

Model V2 removes `WEIGHT` and adds `PASSWORD`, so the two label spaces are not
identical. Reporting one aggregate number across both would be misleading. The
comparison is therefore restricted to the 30 shared entity types, and the full
Model V2 numbers are reported separately alongside it.

In [ ]:
shared_test = restrict_to_labels(test_rows, SHARED_WITH_TINYBERT)
shared_predictions = [
    [span for span in prediction if span["label"] in set(SHARED_WITH_TINYBERT)]
    for prediction in test_predictions
]
shared_metrics = evaluate_spans(shared_test, shared_predictions)
evaluations["test_shared_ontology"] = shared_metrics

print("Shared-ontology comparison (30 entity types, identical AI4Privacy rows)")
print(f"  Model V2 typed entity F1 : {shared_metrics['typed_entity']['f1']:.4f}")
print(f"  Model V2 char recall     : {shared_metrics['private_character_recall']:.4f}")
print(f"  Model V2 leakage         : {shared_metrics['private_character_leakage']:.4f}")
print("\nFrozen TinyBERT IID baseline (same-source synthetic final test, INT8):")
print("  typed entity F1 0.9496 | char recall 0.9969 | leakage 0.0031")
print("\nFull Model V2 ontology (31 types, includes PASSWORD, excludes WEIGHT):")
print(f"  typed entity F1 {test_metrics['typed_entity']['f1']:.4f}")
print("\nThese two Model V2 rows are NOT interchangeable. Only the shared-ontology")
print("row may be compared with TinyBERT; the label spaces differ by WEIGHT and PASSWORD.")

## 21. OOD development set

The generalisation diagnostic. Free to iterate on: error analysis here may guide
data and augmentation design. Reference points from the frozen TinyBERT
baseline, production stack, final partition: typed F1 **0.5665**, character
recall **0.6819**, leakage **0.3181**, hard-negative FP rate **0.6528**.

In [ ]:
ood_dev_metrics, ood_dev_predictions = evaluate_partition("ood_dev", ood_dev, extras=False)

by_category = defaultdict(lambda: ([], []))
for record, prediction in zip(ood_dev, ood_dev_predictions):
    bucket = by_category[str(record.get("category", "unknown"))]
    bucket[0].append(record); bucket[1].append(prediction)

print("\nby category:")
for category, (rows, predictions) in sorted(by_category.items()):
    metrics = evaluate_spans(rows, predictions)
    print(f"  {category:26} typed F1 {metrics['typed_entity']['f1']:.4f} | "
          f"char recall {metrics['private_character_recall']:.4f} | "
          f"hard-neg FP {metrics['hard_negative_false_positive_rate']:.4f}")

print("\nTinyBERT production reference (final partition):")
print("  typed F1 0.5665 | char recall 0.6819 | leakage 0.3181 | hard-neg FP 0.6528")
print("Note: these Model V2 numbers are model-only (no structured regex layer),")
print("so the fairer TinyBERT comparison is its model-only INT8 typed F1 of 0.4649.")

## 22. Frozen OOD final

**Run once, on the selected model, after every other number has been reviewed.**
The frozen set has already been spent once on the current TinyBERT baseline. It
must not inform training data, augmentation, thresholds or checkpoint selection.

Set `RUN_FROZEN_FINAL = True` in section 2 and re-run only this cell.

In [ ]:
if RUN_FROZEN_FINAL:
    ood_final_metrics, _ = evaluate_partition("ood_final", ood_final, extras=False)
    print("\nThe frozen final set has now been used for this checkpoint.")
    print("Do not re-run it for a tuned variant; that would spend it again.")
else:
    print("SKIPPED - RUN_FROZEN_FINAL is False.")
    print("Review sections 17-21 first, then set the flag and re-run this cell only.")

## 23. Save model, tokenizer, config and metrics

In [ ]:
model.save_pretrained(HF_MODEL_DIR, safe_serialization=True)
tokenizer.save_pretrained(TOKENIZER_DIR)

# Match the preflight's compatibility fix: Transformers 5.x serialises
# extra_special_tokens as a list, which the production-pinned 4.53.3 reads as a
# mapping and rejects. Normalise before the artifact leaves this notebook.
tokenizer_config_path = TOKENIZER_DIR / "tokenizer_config.json"
tokenizer_config = json.loads(tokenizer_config_path.read_text(encoding="utf-8"))
if isinstance(tokenizer_config.get("extra_special_tokens"), list):
    tokenizer_config.pop("extra_special_tokens")
    tokenizer_config_path.write_text(
        json.dumps(tokenizer_config, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )
    print("normalised tokenizer_config.json for the production pin")

training_config = {
    "dataset_manifest": str(MANIFEST_PATH.relative_to(REPO_ROOT)),
    "base_model": BASE_MODEL, "base_model_revision": BASE_MODEL_REVISION,
    "num_labels": NUM_LABELS, "entities": list(ENTITIES),
    "seed": SEED, "max_epochs": MAX_EPOCHS, "best_epoch": best_epoch,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "train_batch_size": TRAIN_BATCH_SIZE, "eval_batch_size": EVAL_BATCH_SIZE,
    "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO, "max_grad_norm": MAX_GRAD_NORM,
    "lr_scheduler": "linear warmup then linear decay",
    "mixed_precision": USE_AMP, "context_limit": CONTEXT_LIMIT,
    "overflow_stride": OVERFLOW_STRIDE,
    "train_rows": len(train_rows), "test_rows": len(test_rows),
    "val_original_rows": len(val_original), "val_supplementary_rows": len(val_supplementary),
    "selection_metric": "typed entity F1 on combined validation",
    "best_combined_typed_f1": best_metric,
}
TRAINING_CONFIG_PATH.write_text(json.dumps(training_config, indent=2) + "\n", encoding="utf-8")
(METRICS_DIR / "evaluation_pytorch_fp32.json").write_text(
    json.dumps(evaluations, indent=2, default=float) + "\n", encoding="utf-8"
)
print("saved model, tokenizer, training config and FP32 metrics")

## 24. Export FP32 ONNX

Modern Dynamo exporter, opset 17, dynamic batch and sequence, output
`logits[batch, sequence, 63]`. `PYTHONUTF8=1` is mandatory here - see section 1.

In [ ]:
FP32_ONNX = ONNX_DIR / "model_fp32.onnx"
INT8_ONNX = ONNX_DIR / "model_int8.onnx"

export_model = model.to("cpu").float().eval()
sample = tokenizer(
    ["OnKith Model V2 ONNX export shape check.",
     "A deliberately longer second sentence keeps both exported axes dynamic."],
    return_tensors="pt", padding=True, truncation=True, max_length=CONTEXT_LIMIT,
    return_token_type_ids=False,
)

batch_dim = torch.export.Dim("batch_size", min=1)
sequence_dim = torch.export.Dim("sequence_length", min=2, max=CONTEXT_LIMIT)
onnx_program = torch.onnx.export(
    export_model,
    (sample["input_ids"], sample["attention_mask"]),
    input_names=["input_ids", "attention_mask"], output_names=["logits"],
    opset_version=17, dynamo=True,
    dynamic_shapes={"input_ids": {0: batch_dim, 1: sequence_dim},
                    "attention_mask": {0: batch_dim, 1: sequence_dim}},
)
onnx_program.save(str(FP32_ONNX))

import onnx
graph = onnx.load(str(FP32_ONNX))
onnx.checker.check_model(graph)
print(f"FP32 ONNX: {FP32_ONNX.stat().st_size:,} bytes "
      f"({FP32_ONNX.stat().st_size / 1024**2:.2f} MiB)")
print("inputs :", [i.name for i in graph.graph.input])
print("outputs:", [o.name for o in graph.graph.output])

## 25. ONNX parity

The exported graph must agree with PyTorch before anything is quantized. Both
the raw logits and the decoded spans are compared, at the four sequence lengths
the preflight used.

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(str(FP32_ONNX), providers=["CPUExecutionProvider"])
parity = {"max_abs_logit_delta": {}, "argmax_mismatches": {}}

for length in (75, 221, 289, 512):
    ids = torch.randint(5, 1000, (1, length))
    mask = torch.ones_like(ids)
    with torch.no_grad():
        reference = export_model(input_ids=ids, attention_mask=mask).logits.numpy()
    candidate = session.run(["logits"], {"input_ids": ids.numpy(), "attention_mask": mask.numpy()})[0]
    parity["max_abs_logit_delta"][length] = float(np.abs(reference - candidate).max())
    parity["argmax_mismatches"][length] = int(
        (reference.argmax(-1) != candidate.argmax(-1)).sum()
    )
    print(f"  {length:>4} tokens: max delta {parity['max_abs_logit_delta'][length]:.3e} | "
          f"argmax mismatches {parity['argmax_mismatches'][length]}")

worst = max(parity["max_abs_logit_delta"].values())
assert worst < 1e-4, f"FP32 ONNX diverges from PyTorch (max delta {worst:.3e})"
print(f"\nparity holds: worst delta {worst:.3e} < 1e-4")
(METRICS_DIR / "onnx_parity.json").write_text(json.dumps(parity, indent=2) + "\n", encoding="utf-8")

## 26. INT8 dynamic quantization

Standard ONNX Runtime dynamic quantization, `QInt8`, `per_channel=False`,
`reduce_range=False` - the exact configuration the preflight proved works for
this architecture (49 `MatMulInteger` nodes, roughly 78 MiB).

In [ ]:
from onnxruntime.quantization import QuantType, quantize_dynamic

quantize_dynamic(
    model_input=str(FP32_ONNX), model_output=str(INT8_ONNX),
    weight_type=QuantType.QInt8, per_channel=False, reduce_range=False,
)
int8_graph = onnx.load(str(INT8_ONNX))
onnx.checker.check_model(int8_graph)
operators = Counter(node.op_type for node in int8_graph.graph.node)
fp32_bytes, int8_bytes = FP32_ONNX.stat().st_size, INT8_ONNX.stat().st_size
print(f"FP32 {fp32_bytes:,} bytes ({fp32_bytes / 1024**2:.2f} MiB)")
print(f"INT8 {int8_bytes:,} bytes ({int8_bytes / 1024**2:.2f} MiB)  "
      f"compression {fp32_bytes / int8_bytes:.2f}x")
print("MatMulInteger:", operators.get("MatMulInteger", 0),
      "| DynamicQuantizeLinear:", operators.get("DynamicQuantizeLinear", 0))

## 27. INT8 re-evaluation and quality gate

Export succeeding is not evidence that INT8 is acceptable. The quantized graph
is re-scored on the real evaluation sets and must clear an explicit gate before
it can be deployed.

In [ ]:
from privacy_eval.model_v2_adapter import ModelV2OnnxMasker

int8_masker = ModelV2OnnxMasker(
    INT8_ONNX, TOKENIZER_DIR, {index: label for index, label in ID2LABEL.items()},
    context_limit=CONTEXT_LIMIT, overflow_stride=OVERFLOW_STRIDE,
)


def onnx_predict(rows, masker):
    predictions = []
    for record in rows:
        result = masker.mask_text(record["source_text"])
        predictions.append([
            {"label": entity["entity_type"], "start": entity["start"], "end": entity["end"]}
            for entity in result.entities
        ])
    return predictions


int8_evaluations = {}
for name, rows in (("test", test_rows), ("val_original", val_original),
                   ("val_supplementary", val_supplementary), ("ood_dev", ood_dev)):
    metrics = evaluate_spans(rows, onnx_predict(rows, int8_masker))
    int8_evaluations[name] = metrics
    print(f"{name:20} typed F1 {metrics['typed_entity']['f1']:.4f} | "
          f"leakage {metrics['private_character_leakage']:.4f}")

gate = {}
for name in ("test", "val_original", "val_supplementary"):
    f1_drop = evaluations[name]["typed_entity"]["f1"] - int8_evaluations[name]["typed_entity"]["f1"]
    leak_rise = (int8_evaluations[name]["private_character_leakage"]
                 - evaluations[name]["private_character_leakage"])
    gate[name] = {
        "typed_f1_drop": f1_drop, "leakage_rise": leak_rise,
        "passes": f1_drop < INT8_MAX_TYPED_F1_DROP and leak_rise < INT8_MAX_LEAKAGE_RISE,
    }
    print(f"  gate {name:20} F1 drop {f1_drop:+.4f} | leakage rise {leak_rise:+.4f} | "
          f"{'PASS' if gate[name]['passes'] else 'FAIL'}")

INT8_ACCEPTED = all(entry["passes"] for entry in gate.values())
print(f"\nINT8 quality gate: {'PASSED' if INT8_ACCEPTED else 'FAILED'}")
if not INT8_ACCEPTED:
    print("Do not deploy this INT8 graph. Investigate per-label damage before proceeding.")
(METRICS_DIR / "evaluation_onnx_int8.json").write_text(
    json.dumps({"metrics": int8_evaluations, "gate": gate, "accepted": INT8_ACCEPTED},
               indent=2, default=float) + "\n", encoding="utf-8")

## 28. Production-pin compatibility

The development environment is not the deployment environment. This confirms the
artifacts load under the pinned production versions before anything is shipped.

In [ ]:
continuation = " " + chr(92)   # a trailing backslash, safe inside a raw string
command = [
    r".\model\DeBERTa-v3-xsmall\.venv-production\Scripts\python.exe",
    r"model\DeBERTa-v3-xsmall\verify_production_runtime.py",
    f"--tokenizer {TOKENIZER_DIR.relative_to(REPO_ROOT)}",
    f"--fp32 {FP32_ONNX.relative_to(REPO_ROOT)}",
    f"--int8 {INT8_ONNX.relative_to(REPO_ROOT)}",
    f"--output {(METRICS_DIR / 'production_runtime_compat.json').relative_to(REPO_ROOT)}",
]
print("Run from the repository root, in the isolated production-pin environment:")
print()
for position, part in enumerate(command):
    indent = "  " if position == 0 else "    "
    tail = continuation if position < len(command) - 1 else ""
    print(f"{indent}{part}{tail}")
print()
print("Pins: transformers 4.53.3, tokenizers 0.21.4, onnxruntime 1.22.1, sentencepiece 0.2.1")
print("This must pass before the artifacts are considered deployable.")

## 29. Deployment artifact summary and Raspberry Pi checklist

Development-host measurements are **not** Raspberry Pi measurements. The
preflight measured DeBERTa INT8 at 271 ms for 221 tokens and 849 ms for 512
tokens, single-threaded, on a 14-core Windows host. A 4-core Pi 5 will be
materially slower, so the ~1 s target is plausible around the median and
genuinely uncertain at 512 tokens. Nothing below may be quoted as a Pi result.

In [ ]:
def fingerprint(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return {"path": str(path.relative_to(REPO_ROOT)), "bytes": path.stat().st_size,
            "sha256": digest.hexdigest()}

summary = {
    "ontology": {"entities": len(ENTITIES), "labels": NUM_LABELS,
                 "added": list(ontology.ADDED_ENTITIES),
                 "removed": list(ontology.REMOVED_ENTITIES)},
    "artifacts": {"fp32_onnx": fingerprint(FP32_ONNX), "int8_onnx": fingerprint(INT8_ONNX),
                  "tokenizer": fingerprint(TOKENIZER_DIR / "tokenizer.json")},
    "int8_accepted": INT8_ACCEPTED,
    "headline_metrics": {
        name: {"typed_entity_f1": evaluations[name]["typed_entity"]["f1"],
               "private_character_leakage": evaluations[name]["private_character_leakage"],
               "hard_negative_false_positive_rate":
                   evaluations[name]["hard_negative_false_positive_rate"]}
        for name in ("test", "val_original", "val_supplementary", "combined_validation", "ood_dev")
        if name in evaluations
    },
    "production_migration": {
        "status": "NOT PERFORMED - separate reviewed release step",
        "required_changes": [
            "EXPECTED_LABEL_COUNT 67 -> 63 in integrated_pipeline/pii_adapter.py",
            "pii.label_count 67 -> 63 in integrated_pipeline/config.py and config/live_config.json",
            "ship the DeBERTa tokenizer under FullPipeline_V3/models/",
            "swap the INT8 ONNX artifact",
            "re-run FullPipeline_V3/tests/test_pii_regression.py",
        ],
        "note": "The per-label masking policy is already live and is model-independent.",
    },
    "raspberry_pi_checklist": [
        "Copy the INT8 graph and tokenizer to the Pi 5 (4 GB).",
        "Measure median and p95 latency at 75, 221, 289 and 512 tokens, 1 and 4 threads.",
        "Record RSS during inference, and peak RSS with Canary 180M resident.",
        "Run a thermal soak; report sustained rather than burst latency.",
        "Compare against the ~1 s target at the p95 length of 221 tokens.",
        "Only after these numbers exist may any Pi performance claim be made.",
    ],
}
(METRICS_DIR / "deployment_summary.json").write_text(
    json.dumps(summary, indent=2, default=float) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2, default=float))

---

## Done

Committed evidence: `config/model_v2_label_map.json`,
`config/model_v2_data_manifest.json`, `config/model_v2_source_mapping.csv`,
`config/model_v2_training_config.json`, everything in
`artifacts/model_v2/metrics/`, and `artifacts/model_v2/tokenizer/`.

Local and gitignored: `artifacts/model_v2/data/`, `checkpoints/`,
`huggingface_model/`, `onnx_model/`, and every corpus under `model/data/`.

Not done here, on purpose: the production migration from the TinyBERT 67-label
contract to this 63-label model. That is a separate reviewed release.